# ⚡ BoneRAG — Free Google Colab GPU Backend Deployment

### 📌 Hướng dẫn 3 bước cực đơn giản:
1. **Bật GPU**: Chọn menu `Runtime` -> `Change runtime type` -> Chọn `T4 GPU`.
2. Bấm **Runtime -> Run all (Ctrl + F9)**.
3. Chờ ~60 giây cho đến khi xuất hiện `✅ Server đang chạy!` và copy đường link `https://xxxx.loca.lt` dán vào **Vercel Environment Variable** `VITE_API_BASE_URL`.

In [ ]:
# [Step 1] Kiểm tra GPU NVIDIA & Cài đặt thư viện AI
!nvidia-smi
!pip install -q torch torchvision transformers open_clip_torch faiss-cpu pillow numpy huggingface_hub pyngrok nest_asyncio

In [ ]:
# [Step 2] Tải mã nguồn BoneRAG mới nhất từ GitHub
import os
if not os.path.exists('/content/boneRAG'):
    !git clone https://github.com/thanhnghi-do-2k3/boneRAG.git /content/boneRAG
else:
    %cd /content/boneRAG
    !git pull

%cd /content/boneRAG

In [ ]:
# [Step 3] Khởi chạy Server BoneRAG Backend trên GPU
import subprocess
import time
import urllib.request
import urllib.error

PORT = 8088

# Chạy server BoneRAG ngầm
server_process = subprocess.Popen(
    ['python3', 'demo-app/server.py', '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('⏳ Đang chờ BoneRAG Server khởi động (tải BiomedCLIP + FAISS index)...')
for attempt in range(30):
    time.sleep(3)
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/records')
        print(f'\n✅ Server đang chạy tốt trên port {PORT}! (Thử lần {attempt+1})')
        break
    except Exception:
        print(f'   Lần {attempt+1}/30 — Đang chờ server sẵn sàng...', end='\r')
else:
    print('\n⚠️ Server chưa phản hồi sau 90s. Kiểm tra log bên dưới...')
    out, _ = server_process.communicate(timeout=2)
    print(out.decode('utf-8', errors='replace')[:3000])

In [ ]:
# [Step 4] Mở cổng Public HTTPS Tunnel bằng localtunnel
!npm install -g localtunnel 2>/dev/null
print('\n=======================================================')
print('🌐 ĐƯỜNG LINK BACKEND GPU CÔNG KHAI (copy và dán vào Vercel):')
print('=======================================================')
print('  VITE_API_BASE_URL = <link bên dưới>')
print('\n📌 Lưu ý: Sau khi bấm lưu trên Vercel, bấm Redeploy!')
print('=======================================================')
!npx localtunnel --port 8088